# Supervised classification: readmission inside thirty days

This notebook walks through the classification analysis in the order the
coursework took it: the data and its cleaning, the inputs, the training, and
the results. It calls the same functions as `analysis/m01_classification.py`
and reads the committed tables in `results/` where a step, such as the grid
search, takes longer than a notebook should. The write-up with every table
and figure is [docs/01-supervised-classification.md](../docs/01-supervised-classification.md).

The Diabetes 130-US Hospitals data (Strack et al. 2014) records 101,766
inpatient encounters of diabetic patients at 130 hospitals between 1999 and
2008. The outcome is readmission inside thirty days of discharge. The model is
a feed-forward network with the number of hidden layers among the searched
hyperparameters, and logistic regression is fitted beside it as the reference.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

# Every output directory is redirected to a temporary location before the
# pipeline modules are imported, so this notebook writes nothing into
# results/, figures/ or data/processed/. The committed tables are read from
# results/ directly where the notebook quotes them.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
_scratch = tempfile.mkdtemp()
for _name in ("ML_METHODS_RESULTS", "ML_METHODS_FIGURES", "ML_METHODS_PROCESSED"):
    os.environ[_name] = _scratch
sys.path.insert(0, str(ROOT))

import numpy
import pandas

from src import config, data, evaluate, splits

RESULTS = ROOT / "results"
pandas.set_option("display.width", 120)
pandas.set_option("display.max_columns", 20)


def recorded(prefix=""):
    """The committed metrics record, as a dictionary of strings."""
    table = pandas.read_csv(RESULTS / "metrics.csv", dtype=str)
    return {row.key: row.value for row in table.itertuples()
            if row.key.startswith(prefix)}

## Data and cleaning

`data.load_diabetes` reads the original UCI release committed under
`data/raw/` and applies every cleaning step, counting what each removed.
Encounters ending in death or hospice cannot be readmitted and are removed.
`weight` and `payer_code` are dropped for sparsity, `examide` and
`citoglipton` for taking one value. The admitting department keeps its 16
common levels, the coded categories become strings, the three diagnosis codes
are grouped by the scheme the data's authors published, and age becomes the
midpoint of its bracket.

In [2]:
frame, counts = data.load_diabetes()
pandas.Series(counts)

rows_released                      101766.000000
encounters_released                101766.000000
patients_released                   71518.000000
rows_unobservable_outcome            2423.000000
missing_fraction_weight                 0.968500
missing_fraction_payer_code             0.396600
distinct_values_examide                 1.000000
distinct_values_citoglipton             1.000000
specialty_levels_released              72.000000
specialty_levels_kept                  16.000000
specialty_not_recorded_fraction         0.489400
rows_incomplete                      2235.000000
rows_analyzed                       97108.000000
patients_analyzed                   68166.000000
positive_rate                           0.114573
dtype: float64

In [3]:
print(frame.shape)
frame.head()

(97108, 46)


,encounter_id,patient_nbr,race,gender,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,...,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_30d,diag_1_group,diag_2_group,diag_3_group,age_midpoint
0,2278392,8222157,Caucasian,Female,code 6,code 25,code 1,1,other,41,...,No,No,No,No,No,0,diabetes,missing,missing,5
1,149190,55629189,Caucasian,Female,code 1,code 1,code 7,3,not recorded,59,...,No,No,No,Ch,Yes,0,other,diabetes,other,15
2,64410,86047875,AfricanAmerican,Female,code 1,code 1,code 7,2,not recorded,11,...,No,No,No,No,Yes,0,other,diabetes,other,25
3,500364,82442376,Caucasian,Male,code 1,code 1,code 7,2,not recorded,44,...,No,No,No,Ch,Yes,0,other,diabetes,circulatory,35
4,16680,42519267,Caucasian,Male,code 1,code 1,code 7,1,not recorded,51,...,No,No,No,Ch,Yes,0,neoplasms,neoplasms,diabetes,45


## The partition is drawn over patients

A patient contributes 1.42 encounters on average, so a split drawn over rows
would score the model on patients it had already seen. `splits.grouped_split`
draws the partition over patients with `GroupShuffleSplit` at the recorded
seed, and `splits.report` counts what fell on each side and confirms that no
patient and no identical row sits on both.

In [4]:
LABEL, GROUP = "readmitted_30d", "patient_nbr"
train, test = splits.grouped_split(frame, GROUP)
pandas.Series(splits.report(train, test, GROUP, LABEL))

train_rows              72597.000000
test_rows               24511.000000
test_fraction               0.252400
rows_on_both_sides          0.000000
train_groups            51124.000000
test_groups             17042.000000
groups_on_both_sides        0.000000
train_positive_rate         0.114330
test_positive_rate          0.115295
dtype: float64

## Inputs and the grid search

The model matrix holds 34 categorical and 9 numeric features. The categories
are one-hot encoded and the counts standardized inside the model pipeline, so
the encoder and scaler are fitted on whatever partition the model is fitted
on. The grid of eight architectures and two regularization strengths was
searched under two training regimes, balanced and unbalanced, by three-fold
cross-validation over patients. The search is the largest part of the 1,860
seconds the recorded classification run took, so it is left to the pipeline
script and its table is read here from `results/`.

In [5]:
from analysis import m01_classification as m01

features, categorical, numeric = m01.columns_of(frame)
print(len(categorical), "categorical and", len(numeric), "numeric features")
grid = pandas.read_csv(RESULTS / "m01_grid.csv")
grid.sort_values("cv_roc_auc_mean", ascending=False).reset_index(drop=True)

34 categorical and 9 numeric features


,training,hidden_layers,layer_widths,alpha,cv_roc_auc_mean,cv_roc_auc_sd,mean_iterations
0,unbalanced,3,32-32-32,0.0100,0.658247,0.006927,16.7
1,unbalanced,1,32,0.0100,0.657077,0.012689,19.7
2,unbalanced,2,64-64,0.0100,0.656595,0.001941,18.3
3,unbalanced,1,32,0.0001,0.655988,0.011907,19.3
4,unbalanced,2,64-64,0.0001,0.653912,0.004865,18.3
5,unbalanced,3,32-32-32,0.0001,0.653884,0.009761,21.7
6,unbalanced,2,32-32,0.0100,0.653807,0.008776,13.3
7,unbalanced,2,32-32,0.0001,0.653067,0.008681,13.3
8,balanced,1,32,0.0001,0.616343,0.005926,91.3
9,balanced,1,32,0.0100,0.611837,0.004189,107.0


Every unbalanced configuration scores above every balanced one. The balanced
runs also stop five to nine times later, because the early-stopping slice
under that regime is itself balanced and keeps finding improvement on
duplicated minority rows.

## Training

The selected configuration is read from the committed record and refitted
here on the unbalanced training partition. The identical architecture is
refitted on the balanced training partition, and logistic regression on the
same features and partition. All three are scored on the same 24,511 held-out
encounters at the threshold the pipeline chose on out-of-fold training scores,
0.125. The fits take a minute or two.

In [6]:
import warnings

from sklearn.linear_model import LogisticRegression

# A category present only in the test partition encodes to all zeros, which
# is the intended behavior for a level the training data never held; the
# encoder reports it as a warning and the warning is silenced here.
warnings.filterwarnings("ignore", message="Found unknown categories")

record = recorded("m01.")
parameters = {
    "hidden_layer_sizes": tuple(int(w) for w in record["m01.best_layer_widths"].split("-")),
    "alpha": float(record["m01.best_alpha"]),
}
threshold = float(record["m01.network.test_threshold"])
print("selected:", parameters, "trained", record["m01.best_training"],
      "| threshold", threshold)

balanced = m01.balance(train, config.SEED)
fits = {
    "network, unbalanced, selected": (m01.network(parameters), train),
    "network, balanced": (m01.network(parameters), balanced),
    "logistic regression, unbalanced": (
        LogisticRegression(max_iter=1000, random_state=config.SEED), train),
}
scores = {}
models = {}
for name, (estimator, block) in fits.items():
    model = m01.pipeline(estimator, categorical, numeric)
    model.fit(block[features], block[LABEL])
    models[name] = model
    scores[name] = model.predict_proba(test[features])[:, 1]

selected: {'hidden_layer_sizes': (32, 32, 32), 'alpha': 0.01} trained unbalanced | threshold 0.12473


## Results

In [7]:
columns = ["roc_auc", "average_precision", "brier", "accuracy",
           "balanced_accuracy", "precision", "recall", "f1"]
comparison = pandas.DataFrame(
    {name: evaluate.classification(test[LABEL], score, threshold)
     for name, score in scores.items()}).loc[columns]
comparison.round(4)

,"network, unbalanced, selected","network, balanced","logistic regression, unbalanced"
roc_auc,0.6584,0.5952,0.6582
average_precision,0.2184,0.1568,0.2194
brier,0.0978,0.2338,0.0978
accuracy,0.6468,0.4385,0.7125
balanced_accuracy,0.6102,0.5623,0.6057
precision,0.1764,0.1360,0.1923
recall,0.5626,0.7233,0.4667
f1,0.2686,0.2290,0.2724


The network and logistic regression differ by 0.0002 on ROC-AUC. Balancing the
same architecture costs 0.063 of ROC-AUC and raises the Brier score from 0.098
to 0.234, because a model trained on a population in which half the encounters
end in readmission carries that base rate onto one in which one in nine does.

![ROC and precision-recall curves of the selected network on the held-out encounters.](../figures/fig02_readmission_curves.png)

In [8]:
selected = scores["network, unbalanced, selected"]
predicted = (selected >= threshold).astype(int)
pandas.crosstab(test[LABEL].map({0: "not readmitted", 1: "readmitted"}),
                predicted, rownames=["observed"], colnames=["predicted"])

predicted,0,1
observed,,
not readmitted,14263,7422
readmitted,1236,1590


At the operating point the network flags 9,012 of 24,511 discharges, 36.8
percent, and recovers 1,590 of the 2,826 readmissions among them, at the cost
of 7,422 false alarms.

### The same configuration under repeated partitions

The pipeline redrew the patient-level partition at five seeds and refitted
both models on each. The row at the primary seed reproduces the table above.

In [9]:
repeats = pandas.read_csv(RESULTS / "m01_repeated_splits.csv")
repeats.groupby("model")[["roc_auc", "average_precision"]].agg(["mean", "std", "min", "max"]).round(4)

roc_auc                         average_precision                        
            mean     std     min     max              mean     std     min     max
model                                                                             
logistic  0.6616  0.0066  0.6568  0.6731            0.2187  0.0066  0.2080  0.2260
network   0.6617  0.0085  0.6499  0.6727            0.2189  0.0099  0.2077  0.2319

![Test ROC-AUC and average precision over five patient-level partitions.](../figures/fig12_readmission_repeated_splits.png)

The mean difference between the two models over five draws is 0.0001 of
ROC-AUC, and the network is ahead in three. The standard deviation of 0.007
to 0.009 is the scale for reading the single-partition table: two numbers
closer than about 0.02 are not distinguished by this design.

### The ordering defect the coursework carried

The coursework balanced the classes across the whole cohort and split the
balanced table afterwards, by row. The pipeline fitted the identical
architecture under that ordering and recorded the comparison against the
balanced fit under the proper partition.

In [10]:
pandas.read_csv(RESULTS / "m01_leakage_comparison.csv")

,partition,roc_auc,accuracy,patients_on_both_sides
0,"patients held apart, balanced training",0.595245,0.438456,0
1,"balanced first, split by row",0.805713,0.736852,12057


![Test ROC-AUC of the same balanced network under the two partition orderings.](../figures/fig01_readmission_leakage.png)

The defect adds 0.21 to the reported area. Oversampling before the split
leaves 125,252 pairs of identical rows across the boundary, and splitting by
row places 12,057 patients on both sides. The model is the same in both rows;
what changed is which rows it was scored on.

### What the model uses

Permutation importance on the full test partition, ten permutations per
feature, as the mean fall in ROC-AUC.

In [11]:
pandas.read_csv(RESULTS / "m01_permutation_importance.csv").head(10)

,feature,mean_decrease,sd
0,number_inpatient,0.069283,0.002637
1,discharge_disposition_id,0.031471,0.001709
2,number_emergency,0.006090,0.000989
3,time_in_hospital,0.003813,0.000819
4,diag_1_group,0.003161,0.000933
5,number_diagnoses,0.003002,0.000866
6,age_midpoint,0.002541,0.000583
7,diag_2_group,0.001882,0.000817
8,medical_specialty,0.001867,0.000870
9,diabetesMed,0.001777,0.000509


![The fifteen features whose permutation lowers the held-out ROC-AUC most.](../figures/fig03_readmission_importance.png)

## Discussion

The prior inpatient count and the discharge disposition carry the model, and
nothing else lowers ROC-AUC by more than 0.007 when permuted. The glycated
hemoglobin result the dataset was assembled around is not among the 25
features with the largest decrease. A three-layer network and a linear model
on the same inputs land within 0.0002 of each other on one partition and
within 0.0001 on average over five, so the ceiling near 0.66 belongs to the
recorded facts. Readmission turns on discharge planning, social support,
medication adherence and follow-up care, none of which are in this data.

The one decision that moved the result was balancing. Applied correctly,
inside the training folds, it lowered cross-validated ROC-AUC for every
architecture. Applied before the split, as the coursework did, it reported
0.81 for a model that scores about 0.6 on new patients. The limitations, the
five-partition spread, the single learning rate searched, and the unobserved
readmissions to hospitals outside the network, are set out in the write-up.